# Semaine 10 — Jour 2 : Extraction structurée, valeurs et ambiguïtés

Extraire des champs typés à partir de preuves textuelles conservées.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J2_extraction.ipynb` et `notebooks/week10/teacher/S10_J2_extraction_teacher.ipynb`.

Critère de réussite : Normaliser une durée et un environnement selon un contrat explicite.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Extraire des champs typés à partir de preuves textuelles conservées.
- Distinguer présent, absent et ambigu sans inventer de valeur.
- Normaliser une durée et un environnement selon un contrat explicite.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J2 — Extraction structurée, valeurs et ambiguïtés


## 1. Extraire n’est pas simplement repérer
Un ticket peut mentionner plusieurs codes HTTP ou deux mesures de latence. L’extraction doit produire un objet cohérent avec le contrat attendu, pas seulement toutes les chaînes ressemblant à une valeur. Le lab extrait ticket_id, environment, http_status et duration_ms. Ces champs sont ensuite utilisables par un système, mais aucune action n’est déclenchée.

La méthode obligatoire est une baseline par expressions régulières et normalisation contrôlée. Elle n’est pas un modèle appris. Sur un format stable, cette simplicité peut suffire; sur des formulations variées ou des relations complexes, elle atteint vite ses limites.

## 2. Valeur et preuve
Chaque champ porte status, value et evidence. Une preuve conserve start, end, text et valeur normalisée. La propriété source[start:end] == evidence.text permet d’inspecter ce qui a déclenché l’extraction. Le texte original n’est pas remplacé par une version nettoyée avant de calculer les positions.

Un code HTTP est un entier entre 100 et 599 selon le motif reconnu. Une durée est représentée par une chaîne décimale en millisecondes. Le nombre 1,5 s devient 1500 sans dépendre d’un flottant binaire. L’environnement production est normalisé en prod; les autres valeurs autorisées restent explicites.

## 3. Manquant et ambigu
Aucune occurrence produit missing et value=null. Plusieurs occurrences portant la même valeur peuvent rester present avec plusieurs preuves. Deux valeurs distinctes produisent ambiguous et value=null. Les preuves sont conservées au lieu de sélectionner arbitrairement la première.

Le cas e4 compare HTTP 401 et HTTP 503 et mentionne deux latences. Le contrat ne sait pas quelle occurrence est la bonne mesure cible; il la marque ambiguë. Null ne suffit donc pas pour évaluer le comportement : il faut comparer aussi le statut. Les rapports comptent séparément valeurs et statuts corrects.

## 4. Ce que le motif ne comprend pas
Trouver HTTP 500 dans « aucun HTTP 500 observé » reconnaît une mention, pas une affirmation d’erreur présente. Le champ du lab représente le code cité, sans prétendre inférer une causalité ni un état d’alerte. Une extraction métier d’incident actuel demanderait un contrat plus riche pour la négation, la chronologie et les comparaisons.

Le motif de durée attend latence ou durée, une quantité et une unité s/ms. Une autre écriture peut ne pas être reconnue. Ajouter des formats à partir de cas annotés; ne pas faire évoluer la regex pour réussir un seul exemple sans vérifier les anciens cas.

## 5. Validation d’un objet extrait
Une extraction générative via Structured Outputs pourrait contraindre la forme JSON, mais la forme ne prouverait pas l’appui textuel des valeurs. Conserver preuves, unités et politique d’ambiguïté même si la méthode change. Le lab n’exécute pas cette variante générative.

L’application intégrée demande une revue si l’identifiant de ticket manque, si la classification s’abstient ou si un champ est ambigu. L’absence d’un code HTTP n’est pas forcément un défaut : tous les incidents n’en ont pas. Les champs obligatoires et optionnels relèvent du besoin métier.

## 6. Évaluer
Les quatre textes d’exemple possèdent une annotation de valeur et de statut par champ. Cela représente seize décisions de champ, avec absences et conflits. Un résultat parfait sur ces formats ne mesure pas la robustesse à la langue naturelle générale. Ajouter erreurs d’écriture, unités, négations et chronologie avant d’utiliser la sortie dans un traitement réel.

## À retenir
Une extraction exploitable conserve le lien au texte et expose l’incertitude du contrat. Une valeur manquante et une valeur contradictoire demandent des traitements différents.


## Diagramme du jour

```mermaid
flowchart TD
 A["Texte source"] --> B["Occurrences et offsets"]
 B --> C["Normalisation des valeurs"]
 C --> D{"Valeurs distinctes"}
 D -->|aucune| E["Missing"]
 D -->|une| F["Present"]
 D -->|plusieurs| G["Ambiguous et revue"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import extract, read_data
examples = read_data("examples.json")
for example in examples:
    result = extract(example["text"])
    for key, field in result.items():
        assert field["value"] == example["expected"][key]
        assert field["status"] == example["expected_status"][key]
        assert all(example["text"][e["start"]:e["end"]] == e["text"] for e in field["evidence"])
print("16 valeurs et 16 statuts vérifiés")

# Lab J2 — Extraction structurée, valeurs et ambiguïtés

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day02/labs/run.py
python3 -m examples.week10.demo --day 2
```

Résultat attendu : **16 valeurs et 16 statuts comparés, avec deux champs ambigus dans e4**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Convertir une durée

Normaliser 1,5 s, 250 ms et 0,25 ms sans inventer une précision entière.

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Lire un conflit

Un texte contient deux fois HTTP 503 puis HTTP 401. Quel statut produire ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Interpréter une mention

La phrase aucun HTTP 500 observé prouve-t-elle une erreur 500 active ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Pourquoi garder evidence avec la valeur ?

## Question 2

Quelle différence entre JSON valide et extraction correcte ?


# Challenge

Créer un ticket sans durée, un autre avec deux durées identiques et un troisième avec deux durées différentes. Vérifier missing, present et ambiguous et les offsets de chaque preuve.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Références — Jour 2

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
